# Simulation Sanity Check

Load a saved simulation pickle, validate its numerical and process-level outputs, and inspect standard figures.

## Execution paths

1. **Inspection (default):** load the selected pickle and run every following section without changing it.
2. **Optional re-run:** set `RERUN_SIMULATION = True`, define any parameter overrides in the dedicated section, and re-run the loaded configuration before validation.

Run the notebook from the campaign directory so pickle discovery starts at the correct location. The selected pickle should contain detailed trajectories; performance-mode results can lack data required by some plots.

**Workflow:**
1. Setup and configuration
2. Load or optionally re-run the simulation
3. KPI, feasibility, and elemental-closure diagnostics
4. Batch, kinetic, and process-level visualizations
5. Interpretation checklist

## 1. Setup and Configuration

In [1]:
from pathlib import Path

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.simulation_data import SimulationData

print("✓ Imports successful")

✓ Imports successful


### 1.1 Discover Available Simulation Pickles

Scan the campaign directory recursively for saved simulation pickles. Configure the selected result and inspection scope in the next cell.

In [2]:
# ── Scan this campaign directory recursively for simulation pickles ───────────
CAMPAIGN_DIR = Path.cwd()
PICKLE_FILES = sorted(
    path for path in CAMPAIGN_DIR.rglob("*.pkl")
    if path.is_file()
)

if not PICKLE_FILES:
    raise FileNotFoundError(
        f"No .pkl files found below the campaign directory: {CAMPAIGN_DIR.resolve()}"
    )

print(f"Found {len(PICKLE_FILES)} simulation pickle(s) below {CAMPAIGN_DIR.resolve()}:\n")
for number, path in enumerate(PICKLE_FILES, start=1):
    print(f"[{number:>3}] {path.relative_to(CAMPAIGN_DIR)}")


Found 25 simulation pickle(s) below C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics:

[  1] 01_FedBatch\export\fedbatch_constant_p2_20260819_0951\opt_20260819_0951.pkl
[  2] 01_FedBatch\export\fedbatch_constant_p2_20260824_2340\opt_20260824_2340.pkl
[  3] 01_FedBatch\export\fedbatch_csfr_p2_20260819_1106\opt_20260819_1106.pkl
[  4] 01_FedBatch\export\fedbatch_csfr_p2_20260825_0001\opt_20260825_0001.pkl
[  5] 01_FedBatch\export\fedbatch_exponential_p2_20260819_1232\opt_20260819_1232.pkl
[  6] 01_FedBatch\export\fedbatch_exponential_p2_20260825_0025\opt_20260825_0025.pkl
[  7] 01_FedBatch\export\fedbatch_gsp_p2_20260819_1707\opt_20260819_1707.pkl
[  8] 01_FedBatch\export\fedbatch_gsp_p2_20260825_0135\opt_20260825_0135.pkl
[  9] 01_FedBatch\export\fedbatch_linear_p2_20260819_1833\opt_20260819_1833.pkl
[ 10] 01_FedBatch\export\fedbatch_linear_p2_20260825_0204\opt_20260825_0204.pkl
[ 11] 02_Batch\export\batch_p2_20260

In [3]:
# ── Select the saved simulation and inspection scope ──────────────────────────
# Set this to one of the numbers printed by the preceding discovery cell.
PICKLE_NUMBER = 2

# Individual batch used by the batch and kinetic figure sections.
BATCH_NUMBER = 1

# None includes every batch in process-level overlays; use an integer to limit them.
MAX_BATCHES = None

# False preserves the loaded result. Set True only to apply overrides and re-run.
RERUN_SIMULATION = False

if not isinstance(PICKLE_NUMBER, int) or isinstance(PICKLE_NUMBER, bool):
    raise TypeError("PICKLE_NUMBER must be an integer.")
if not 1 <= PICKLE_NUMBER <= len(PICKLE_FILES):
    raise ValueError(
        f"PICKLE_NUMBER must be between 1 and {len(PICKLE_FILES)}; "
        f"received {PICKLE_NUMBER}."
    )
if not isinstance(BATCH_NUMBER, int) or isinstance(BATCH_NUMBER, bool) or BATCH_NUMBER < 1:
    raise ValueError("BATCH_NUMBER must be a positive integer.")
if MAX_BATCHES is not None and (
    not isinstance(MAX_BATCHES, int) or isinstance(MAX_BATCHES, bool) or MAX_BATCHES < 1
):
    raise ValueError("MAX_BATCHES must be None or a positive integer.")

PICKLE_PATH = PICKLE_FILES[PICKLE_NUMBER - 1]

print(f"Selected pickle [{PICKLE_NUMBER}]: {PICKLE_PATH.relative_to(CAMPAIGN_DIR)}")
print(f"Batch for detailed plots: {BATCH_NUMBER}")
print(f"Optional re-run enabled: {RERUN_SIMULATION}")

Selected pickle [2]: 01_FedBatch\export\fedbatch_constant_p2_20260824_2340\opt_20260824_2340.pkl
Batch for detailed plots: 1
Optional re-run enabled: False


## 2. Load the Selected Simulation

This is the default inspection path. The optional re-run section below replaces `sim` and `data` only when enabled.

In [4]:
sim = CellCultureSimulation.load_pickle(PICKLE_PATH, base_export_dir=PICKLE_PATH.parent)
data = sim.data

if not 1 <= BATCH_NUMBER <= len(data.batches):
    raise ValueError(
        f"BATCH_NUMBER must be between 1 and {len(data.batches)} for the selected simulation; "
        f"received {BATCH_NUMBER}."
    )

print(f"✓ Loaded: {data.simulation_type}")
print(f"  Source           : {PICKLE_PATH.resolve()}")
print(f"  Batches          : {len(data.batches)}")
print(f"  Simulation time  : {data.simulated_window_totals_stats.get('total_simulation_time_h', 'N/A'):.1f} h")
print(f"  Parameters name  : {data.parameters.metadata.get('name', 'N/A') if data.parameters else 'N/A'}")

CellCultureSimulation loaded from: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics\01_FedBatch\export\fedbatch_constant_p2_20260824_2340\opt_20260824_2340.pkl
✓ Loaded: Fedbatch
  Source           : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics\01_FedBatch\export\fedbatch_constant_p2_20260824_2340\opt_20260824_2340.pkl
  Batches          : 11
  Simulation time  : 960.0 h
  Parameters name  : base_parameters


### 2.1 Optional Parameter Overrides and Re-run

Leave `RERUN_SIMULATION = False` to inspect the loaded result unchanged. To explore a changed configuration, enable it in the configuration cell and add parameter-path/value pairs to `PARAMETER_OVERRIDES` in the next cell. The re-run replaces `sim` and `data` for every subsequent diagnostic and figure.

### 2.2 Run Optional Overrides

In [5]:
# Keep this mapping empty to re-run the saved configuration without changes.
# Uncomment or add entries only when RERUN_SIMULATION is True.
PARAMETER_OVERRIDES = {
    # "fed_batch.C_Gln_feed": 10,
    # "fed_batch.C_AA_feed": 10,
    # "initial_conditions.C_Glc_init": 40,
    # "initial_conditions.C_Gln_init": 20,
    # "fed_batch.CSFR": 5e-12,
    # "perfusion.CSPR": 10.0e-11,
}

if RERUN_SIMULATION:
    for parameter_path, value in PARAMETER_OVERRIDES.items():
        sim.parameters[parameter_path] = value

    sim = CellCultureSimulation(
        parameters=sim.parameters,
        kinetics=sim.kinetics,
        gas_controller=sim.gas_controller,
        operating_mode=sim.operating_mode,
        feed_strategy=sim.feed_strategy,
        perfusion_strategy=sim.perfusion_strategy,
        stoichiometry_mode=sim.stoichiometry_mode,
        performance_mode=sim.performance_mode,
        solver_method="LSODA",
    )
    data = sim.run_simulation()
    print(f"✓ Re-ran simulation with {len(PARAMETER_OVERRIDES)} override(s).")
else:
    print("✓ Inspection mode: using the loaded simulation unchanged.")

✓ Inspection mode: using the loaded simulation unchanged.


## 3. Validation and Diagnostics

Review optimization feasibility, performance, and elemental closure before interpreting figures.

### 3.1 Optimization Feasibility Check

This uses the same `ObjectiveFunction` path and rejection criteria as differential evolution. A pass means the result is feasible for optimization; it does not replace scientific review.

In [6]:
# Evaluate the current simulation with the same ObjectiveFunction implementation
# used by differential evolution. This avoids maintaining a second, drifting copy
# of the optimization rejection logic in the notebook.

from animakinetica.analysis.optimization import ObjectiveFunction
import numpy as np

WALL_TIMEOUT_S = 10.0  # Match the campaign optimization timeout when diagnosing runtime rejections.

# ObjectiveFunction needs objectives and weights even though this cell only
# evaluates feasibility. Use a harmless available numeric KPI with zero weight.
validation_function = ObjectiveFunction(
    baseline_simulation=sim,
    opt_config={},
    objectives=["STY_kg_m3_day"],
    weights=[0.0],
    timeout=WALL_TIMEOUT_S,
    baseline_values=None,
    debug_rejections=True,
)

# An empty optimization configuration maps this empty vector to the current
# baseline parameters, then runs the same temporary-simulation path and every
# ObjectiveFunction rejection check. Valid evaluations return a non-positive
# score; soft constraint penalties and hard rejections return positive scores.
filter_score = float(validation_function(np.array([], dtype=float)))
passes_filter = filter_score <= 0.0

print("ObjectiveFunction filter check:")
print(f"  {'PASS' if passes_filter else 'FAIL'}  valid ObjectiveFunction score <= 0")
print(f"  Score: {filter_score:.6g}")

# Supplementary diagnostics only; all pass/fail decisions remain in
# ObjectiveFunction. The fields below expose inputs to its current filters.
simulated_totals = data.simulated_window_totals_stats
expected_window_h = validation_function._get_expected_simulation_window_target_h()
soft_violations = validation_function._get_soft_constraint_violations(
    data,
    simulated_totals,
)

try:
    closure = data.compute_boundary_closure()
    elem_res = {
        element: abs(values["relative_residual_pct"])
        for element, values in closure["elements"].items()
    }
except (ValueError, KeyError):
    elem_res = None

total_base_in_L = float(simulated_totals.get("total_base_in_L", 0.0) or 0.0)
total_acid_in_L = float(simulated_totals.get("total_acid_in_L", 0.0) or 0.0)
total_medium_in_L = float(simulated_totals.get("total_medium_in_L", 0.0) or 0.0)
acid_base_fraction = (
    (total_base_in_L + total_acid_in_L) / total_medium_in_L
    if total_medium_in_L > 0.0 else None
)

print("\nDiagnostics:")
print(f"  total_simulation_time_h = {float(simulated_totals.get('total_simulation_time_h', 0.0) or 0.0):.3f}")
print(f"  expected_simulation_window_h = {expected_window_h:.3f}")
print(f"  total_active_cultivation_time_h = {float(simulated_totals.get('total_active_cultivation_time_h', 0.0) or 0.0):.3f}")
print(f"  projection_window_h = {simulated_totals.get('projection_window_h')}")
print(f"  projection_target_time_h = {simulated_totals.get('projection_target_time_h')}")
print(f"  number_of_batches = {simulated_totals.get('number_of_batches')}")
print(f"  total_DCM_kg = {simulated_totals.get('total_DCM_kg', data.summary_stats.get('total_DCM_kg'))}")
print(f"  last_batch_termination_reason = {simulated_totals.get('last_batch_termination_reason')}")
print(f"  unique_batch_termination_reasons = {simulated_totals.get('unique_batch_termination_reasons')}")
print(f"  batch_final_volume_min_fraction = {simulated_totals.get('batch_final_volume_min_fraction')}")
print(f"  batch_final_volume_max_fraction = {simulated_totals.get('batch_final_volume_max_fraction')}")
print(f"  last_batch_final_volume_fraction = {simulated_totals.get('last_batch_final_volume_fraction')}")
print(f"  total_base_in_L = {total_base_in_L:.6g}")
print(f"  total_acid_in_L = {total_acid_in_L:.6g}")
print(f"  total_medium_in_L = {total_medium_in_L:.6g}")
print(f"  acid_base_fraction = {acid_base_fraction if acid_base_fraction is not None else 'N/A'}")
print(f"  phase_active_runtime_h = {simulated_totals.get('phase_active_runtime_h')}")
print(f"  batch_phase_active_runtime_h = {simulated_totals.get('batch_phase_active_runtime_h')}")
print(f"  ObjectiveFunction soft_constraint_violations = {soft_violations}")
if elem_res is not None:
    print(
        "  boundary closure |relative_residual_pct| "
        f"(soft limit {ObjectiveFunction._MAX_CLOSURE_PCT:.0f}%) = {elem_res}"
    )

run_simulation execution time: 330.446 ms
ObjectiveFunction filter check:
  PASS  valid ObjectiveFunction score <= 0
  Score: -0

Diagnostics:
  total_simulation_time_h = 960.000
  expected_simulation_window_h = 960.000
  total_active_cultivation_time_h = 0.000
  projection_window_h = 960.0
  projection_target_time_h = 7680.0
  number_of_batches = 11
  total_DCM_kg = None
  last_batch_termination_reason = Maximum time reached
  unique_batch_termination_reasons = ['Max volume', 'Maximum time reached']
  batch_final_volume_min_fraction = 0.9840299698636074
  batch_final_volume_max_fraction = 1.0000000500000001
  last_batch_final_volume_fraction = 0.9840299698636074
  total_base_in_L = 0
  total_acid_in_L = 0
  total_medium_in_L = 18206.4
  acid_base_fraction = 0.0
  phase_active_runtime_h = {'BATCH': 61.1726695404494, 'FEDBATCH': 898.8273304595506}
  batch_phase_active_runtime_h = [{'batch_number': 1, 'BATCH': 46.23199643881059, 'FEDBATCH': 81.88140048775993}, {'batch_number': 2, 'BATCH'

### 3.2 Performance Summary

Inspect the primary KPIs and cultivation-level totals.

In [7]:
print("Performance Metrics:")
for key, value in data.summary_stats.items():
    if isinstance(value, float):
        print(f"  {key:<45} {value:.4f}")
    else:
        print(f"  {key:<45} {value}")

Performance Metrics:
  total_DCM_harvested_kg                        69.5924
  total_WCM_harvested_kg                        463.9492
  STY_kg_m3_day                                 0.1087
  STY_WCM_kg_m3_day                             0.7249
  medium_conversion_efficiency_m3_kg            2.0929
  medium_conversion_efficiency_m3_kg_WCM        0.3139
  protein_conversion_efficiency_kg_kg           28.6785
  feed_conversion_efficiency_kg_kg              23.1543
  feed_conversion_efficiency_kg_kg_WCM          3.4731


In [8]:
# Cultivation summary table
data.get_cultivation_summary_df()

,Batch No.,Termination Reason,Cultivation Time (h),Batch Start Time (h),Batch End Time (h),Accum. Cultivation Time (d),Final Glucose (mmol/L),Final Glutamine (mmol/L),Final Lactate (mmol/L),Final Ammonia (mmol/L),...,Accum. Acid Addition Use (L),Accum. Total Inlet Use (L),Harvest Volume (L),Accum. Harvest Volume (L),Final Cell Density (10^6 cells/mL),Final Cell Density (g/L),Cells Harvested (10^12 Cells),Cells Harvested (kg),Accum. Cell Harvest (10^12 Cells),Accum. Biomass Harvest (kg)
0,1,Max volume,128.113397,0.000000,128.113397,5.338058,1.327610,46.239832,14.847489,6.048956,...,0.0,2000.000163,1654.193893,1654.193893,3.470152,0.520523,5.740304,0.861046,5.740304,0.861046
1,2,Max volume,84.836970,128.113397,212.950367,8.872932,1.433364,45.174646,15.497990,6.633146,...,0.0,3654.194072,1635.335962,3289.529856,3.290699,0.493605,5.381399,0.80721,11.121703,1.668255
2,3,Max volume,83.744971,212.950367,296.695338,12.362306,1.594728,45.048212,15.708419,6.581124,...,0.0,5289.530035,1621.458649,4910.988505,3.170062,0.475509,5.140125,0.771019,16.261828,2.439274
3,4,Max volume,83.226353,296.695338,379.921691,15.830070,1.637062,44.999645,15.813958,6.549049,...,0.0,6910.988687,1618.342484,6529.330989,3.144179,0.471627,5.088359,0.763254,21.350187,3.202528
4,5,Max volume,83.135735,379.921691,463.057426,19.294059,1.641276,44.980118,15.850391,6.544228,...,0.0,8529.331173,1618.104252,8147.435241,3.142218,0.471333,5.084436,0.762665,26.434623,3.965193
5,6,Max volume,83.132348,463.057426,546.189773,22.757907,1.640661,44.974837,15.858765,6.544465,...,0.0,10147.435428,1618.166738,9765.601979,3.142732,0.471410,5.085465,0.76282,31.520088,4.728013
6,7,Max volume,83.134967,546.189773,629.324740,26.221864,1.640332,44.973839,15.860064,6.544761,...,0.0,11765.602167,1618.192312,11383.794291,3.142943,0.471441,5.085886,0.762883,36.605973,5.490896
7,8,Max volume,83.135766,629.324740,712.460506,29.685854,1.640271,44.973712,15.860178,6.544838,...,0.0,13383.794482,1618.196385,13001.990677,3.142976,0.471446,5.085953,0.762893,41.691926,6.253789
8,9,Max volume,83.135864,712.460506,795.596370,33.149849,1.640269,44.973705,15.860177,6.544847,...,0.0,15001.990869,1618.196382,14620.187058,3.142976,0.471446,5.085953,0.762893,46.777879,7.016682
9,10,Max volume,83.135856,795.596370,878.732225,36.613843,1.640271,44.973705,15.860177,6.544847,...,0.0,16620.187253,1618.196206,16238.383264,3.142975,0.471446,5.085950,0.762892,51.863829,7.779574


In [9]:
process_stoich_closure = data.compute_stoichiometric_closure()
process_stoich_closure_df = data.build_closure_summary_df(process_stoich_closure)

process_stoich_biomass = next(
    spec for spec in process_stoich_closure['component_specs']
    if spec['label'] == 'Dry biomass'
)

print('Integrated stoichiometric CHON closure:')
display(process_stoich_closure_df)
print(f"Max absolute elemental residual: {process_stoich_closure['max_abs_residual']:.6g} mol-atoms")

Integrated stoichiometric CHON closure:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,1904.132265,1904.117954,-0.014311,-0.000752
1,H,3823.397067,3823.363594,-0.033473,-0.000875
2,O,2838.482001,2838.472533,-0.009468,-0.000334
3,N,203.250325,203.245480,-0.004846,-0.002384


Max absolute elemental residual: 0.0334728 mol-atoms


In [10]:
process_external_elemental_closure = data.compute_boundary_closure()
process_external_elemental_closure_df = data.build_closure_summary_df(process_external_elemental_closure)

process_dry_biomass = next(
    spec for spec in process_external_elemental_closure['component_specs']
    if spec['label'] == 'DCM out'
 )

print('Whole-process external-boundary CHON closure:')
display(process_external_elemental_closure_df)
print(f"Max absolute elemental residual: {process_external_elemental_closure['max_abs_residual']:.6g} mol-atoms")

Whole-process external-boundary CHON closure:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,7.564002e+03,7.563989e+03,-0.013235,-0.000175
1,H,2.035401e+06,2.035401e+06,-0.032074,-0.000002
2,O,1.017418e+06,1.017417e+06,-0.031047,-0.000003
3,N,2.118836e+03,2.118831e+03,-0.004834,-0.000228


Max absolute elemental residual: 0.0320739 mol-atoms


## 4. Batch-State and Operating-Control Figures

Use `BATCH_NUMBER` from the configuration section to inspect biomass, metabolites, volume/feed, gas transfer, and pH control for one batch.

## 5. Batch Plots

In [11]:
# Cell density & metabolites
fig = data.plot_biomass_metabolites(batch_number=BATCH_NUMBER)
fig.show()

In [12]:
# Volume & feed / dilution rate
fig = data.plot_volume_feed(batch_number=BATCH_NUMBER)
fig.show()

In [13]:
# Gas transfer (DO, agitation, gas flow, O2 fraction)
fig = data.plot_gas_transfer(batch_number=BATCH_NUMBER)
fig.show()

In [14]:
# pH plots
fig = data.plot_pH_control(batch_number=BATCH_NUMBER)
fig.show()

## 5. Kinetic-Behavior Figures

Review nutrient limitation, inhibition, kinetic rates, and stoichiometric yields for the selected batch.

In [15]:
# Nutrient limitation factors (Monod terms)
fig = data.plot_nutrient_factors(batch_number=BATCH_NUMBER)
fig.show()

In [16]:
# Inhibition factors (lactate, ammonia, O2, osmolality)
fig = data.plot_inhibition_factors(batch_number=BATCH_NUMBER)
fig.show()

In [17]:
# Specific rates (mu, q_Glc, q_Gln, q_Lac, q_O2, etc.)
fig = data.plot_kinetic_rates(batch_number=BATCH_NUMBER)
fig.show()

In [18]:
# Stoichiometric yield coefficients (Y_X/Glc, Y_Lac/Glc, Y_O2/Glc, etc.)
fig = data.plot_stoichiometric_yields(batch_number=BATCH_NUMBER)
fig.show()

## 6. Multi-Batch Process Figures

Use `MAX_BATCHES` from the configuration section to constrain batch overlays and cumulative totals.

In [19]:
# Overlaid state variable profiles across batches
figs = data.plot_profiles(max_batches=MAX_BATCHES)
for fig in figs:
    fig.show()

In [20]:
# Cumulative harvest (biomass & volume) across batches
fig = data.plot_cumulative_harvest(max_batches=MAX_BATCHES)
fig.show()

In [21]:
# Cumulative medium use
fig = data.plot_cumulative_medium_use(max_batches=MAX_BATCHES)
fig.show()

In [22]:
# Cumulative gas use
fig = data.plot_cumulative_gas_use(max_batches=MAX_BATCHES)
fig.show()

## 7. Interpretation Checklist

Before accepting a result:

- Confirm the selected pickle and whether the optional re-run path was enabled.
- Investigate a failed optimization feasibility check, especially its rejection diagnostics and termination reason.
- Review elemental residuals in both closure reports; large residuals require mass-balance investigation.
- Check the selected-batch figures for unexpected depletion, volume limits, gas-control saturation, or pH-control behavior.
- If a plot lacks data, verify that the pickle was generated with `performance_mode=False` and contains detailed trajectories.